<p style="text-align: center">
<img src="../../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>


# Integration

You've trained a YOLOv5 model and placed `best.onnx` under `tasks/object_detection/models/`. Now you'll run a live server that:

1. Streams the simulator's camera feed.
2. Runs your model on each frame.
3. Filters predictions through the rules you write in [`integration_activity.py`](../../packages/integration_activity.py).
4. Displays bounding boxes, labels, and confidence scores in a web UI.

This notebook walks through the filters and how to launch the server.


## 1. Class IDs

Your model was trained on three classes:

| ID | Object |
|----|--------|
| 0  | Duckie |
| 1  | Truck  |
| 2  | Sign   |

Trucks are always parked off the road. Signs are static. Only **duckies** are pedestrians the bot must avoid — so most of the time, you will filter everything else out.


## 2. Filtering by class

Edit `filter_by_classes` in [`integration_activity.py`](../../packages/integration_activity.py). Return `True` to keep a prediction, `False` to drop it.

A common starting point — keep only duckies:

```python
def filter_by_classes(pred_class: int) -> bool:
    return pred_class == 0
```


## 3. Filtering by confidence

Low-confidence predictions are often false positives. Edit `filter_by_scores` to drop them:

```python
def filter_by_scores(score: float) -> bool:
    return score >= 0.6
```

The right threshold depends on how your model trained. Start at `0.5` and tune from there — too low and you'll get spurious boxes, too high and you'll miss real duckies.


## 4. Filtering by bounding box

Even a correct duckie detection might not matter — a tiny bounding box at the edge of the frame is probably a duckie far off the road, not in the bot's path.

<p style="text-align:center;"><img src="../../../assets/images/thirds.png" width="450"></p>

Use `filter_by_bboxes` to drop boxes that are too small or too far to the side. The bbox is `(xmin, ymin, xmax, ymax)` in pixels:

```python
def filter_by_bboxes(bbox):
    xmin, ymin, xmax, ymax = bbox
    width  = xmax - xmin
    height = ymax - ymin
    area   = width * height
    return area > 800
```


## 5. Frame skipping

YOLOv5 inference is expensive — running it on every frame can drop your framerate. If your model is too slow, increase the skip count in `NUMBER_FRAMES_SKIPPED`:

```python
def NUMBER_FRAMES_SKIPPED() -> int:
    return 2  # run inference every 3rd frame
```

Skipped frames reuse the previous detection result, so the boxes flicker less than you'd think.


## 6. Run the server

From the project root, launch the simulator with the object detection task:

```bash
python launch.py --sim --task object_detection
```

The terminal will print a URL like `http://localhost:5000`. Open it in a browser to see the live camera feed with your model's bounding boxes drawn on top.

The sidebar shows:
- **Status** — model loaded successfully or the error if not.
- **Configuration** — input size, confidence threshold, NMS IoU.
- **Live detections** — every box the model returns this frame, after your filters.

Edit `integration_activity.py`, restart the server, and watch the boxes change.


## 7. Tuning loop

1. Run the server.
2. Drive past some objects in the simulator (or re-spawn).
3. Watch which boxes appear in the sidebar.
4. Update your filters in `integration_activity.py`.
5. Restart the server, repeat.

There's no objectively correct answer — robotics is iterative. The goal is reliable duckie detection with few false positives. Trust your filters.


## 8. Your task - implement `should_stop`

Now that your filters are tuned, open [`tasks/object_detection/packages/stop_activity.py`](../../packages/stop_activity.py) 
and implement the `should_stop` function.

The bot calls this every frame with the list of detections that passed your filters. 
Return `(True, reason)` to stop the bot, or `(False, "")` to keep moving.

### Function signature

```python
def should_stop(detections: List[Detection], img_size: int) -> Tuple[bool, str]:
```

- `detections` — list of `((x1, y1, x2, y2), score, class_id)` tuples that passed your filters
- `img_size` — the height/width of the camera frame in pixels

### What to think about

A detection being present does not necessarily mean the bot should stop — 
a duckie far away (small box, high in the frame) is not an immediate threat. 
A duckie close ahead (large box, low in the frame) is.

Some things to consider:

- **Vertical position** — `y2` is the bottom edge of the box. A large `y2` means the object is close.
- **Box size** — a large bounding box means the object is near.
- **Proximity threshold** — you might stop only when `y2 > img_size * 0.6`, for example.

Tune the threshold by running the server, driving toward a duckie, and watching when the bot stops.
